# Cost Management & Azure Pricing

Azure charges for resources based on consumption — you pay for what you use, when you use it. Understanding the pricing model and actively managing costs is essential for any Azure deployment.

## How Azure Charges Work

| Dimension | Examples |
|---|---|
| **Compute** | Per hour/second for VMs, App Service plans, AKS nodes |
| **Storage** | Per GB stored + per operation (reads/writes/lists) |
| **Networking** | Egress (outbound data leaving Azure) charged per GB; ingress free |
| **Managed services** | Per DTU/vCore-hour (SQL), per RU/s (Cosmos DB), per message (Service Bus) |
| **Support plans** | Developer, Standard, Professional Direct, Premier |

Key principle: **egress costs money, ingress is free**. Data leaving Azure to the internet or to another region is charged. Data entering Azure and data moving within the same region between services is generally free.

## Pricing Tools

- **Azure Pricing Calculator** — estimate monthly costs before deploying. Configure services, regions, and usage patterns to get a cost estimate.
- **Azure TCO Calculator** — Total Cost of Ownership comparison between on-premises infrastructure and Azure. Accounts for hardware, software licences, facilities, and IT labour.
- **Cost Management + Billing** — the portal experience for viewing actual costs, setting budgets, and analysing spending trends.

## Azure Cost Management

Azure Cost Management (now part of Microsoft Cost Management) provides visibility, analysis, and control over Azure spending.

### Cost Analysis

Cost Analysis lets you visualise spending broken down by:
- **Scope**: Management group, subscription, resource group, or resource
- **Time period**: Daily, monthly, or custom date ranges
- **Grouping dimensions**: Resource type, service name, location, resource group, tag

Views include accumulated cost (spend to date vs budget), daily cost (spending rate), cost by service, and cost by resource. You can save custom views and share them with your team.

### Budgets

A **budget** sets a spending threshold and triggers alerts when actual or forecasted spend reaches a percentage of that threshold:

- Scope: management group, subscription, or resource group
- Reset period: monthly, quarterly, or annually
- Alert conditions: actual spend at 80%, actual spend at 100%, forecasted spend at 110%
- Actions: email notification, or trigger an Action Group (which can call an Azure Function or Logic App for automated response — e.g. shut down dev VMs)

### Cost Alerts

| Alert type | Triggers when |
|---|---|
| Budget alert | Spend crosses a budget threshold |
| Credit alert | Azure credit balance falls below a threshold (EA/MCSP) |
| Department spending quota alert | Department reaches a % of its quota (EA only) |
| Anomaly alert | Unusual spending pattern detected by ML |

### Cost Exports

Schedule automated exports of cost data to an Azure Storage Account in CSV format. Common uses: feed data into Power BI for custom dashboards, import into internal cost allocation systems, or archive for compliance.

## Pricing Models — Compute

Azure offers several purchasing models for compute that dramatically affect cost:

### Pay-as-you-go (PAYG)
Full list price, billed per second. No commitment. Maximum flexibility — use for unpredictable or short-lived workloads.

### Reserved Instances (RIs)
Commit to a specific VM size/region for **1 or 3 years** in exchange for up to **72% discount** over PAYG.

- **Scope**: shared (applies to any VM in the subscription/billing account that matches) or single subscription
- **Flexibility**: Instance size flexibility lets one RI cover VMs of different sizes within the same instance series (e.g. D2s_v5 ↔ D4s_v5)
- **Payment**: upfront (max discount), monthly (no extra cost vs upfront for 1-year), or no upfront
- RIs are available for VMs, SQL Database, Cosmos DB, App Service, AKS nodes, and more
- Best for: steady-state workloads running 24/7

### Savings Plans
Commit to a **fixed hourly spend** (e.g. $10/hour) across any eligible compute for 1 or 3 years — up to **65% discount**.

- More flexible than RIs: covers VMs, App Service, Functions Premium, AKS, Azure Spring Apps
- Not tied to a specific VM size or region
- Best for: workloads that scale or change VM sizes frequently

### Spot VMs
Use Azure's unused compute capacity at up to **90% discount**. Azure can evict Spot VMs with 30-second notice when capacity is needed.

- Best for: fault-tolerant, interruptible workloads — batch processing, CI/CD agents, rendering, ML training
- Not suitable for production web servers or databases

### Azure Hybrid Benefit (AHB)
Use existing **Windows Server or SQL Server licences** (with Software Assurance) on Azure VMs — saves up to **85%** on Windows VM costs.

- Also applies to Azure SQL (SQL Database, SQL MI, SQL on IaaS)
- Linux VMs: AHB lets you convert a RHEL or SLES pay-as-you-go VM to bring-your-own-subscription pricing

### Combining discounts

Discounts stack:
- RI + Azure Hybrid Benefit → up to ~80% off list price for Windows VMs
- Dev/Test pricing (Visual Studio subscribers) → significant discounts on VMs and services for non-production use

## Cost Optimisation Strategies

### Right-sizing
Many workloads are over-provisioned. Azure Advisor analyses VM utilisation metrics and recommends downsizing or shutting down underutilised VMs. A VM consistently below 5% CPU utilisation is a candidate for a smaller SKU.

### Auto-shutdown and scaling
- **Dev/test VMs**: configure auto-shutdown at 7 PM and require manual start — a VM that runs 12 hours instead of 24 costs 50% less
- **VMSS / AKS**: autoscaling ensures you only pay for capacity when demand requires it
- **App Service**: scale down to fewer instances (or to Free/Shared tier) outside business hours
- **Azure SQL serverless**: automatically pauses when idle and resumes on next query — ideal for intermittent workloads

### Storage optimisation
- Use **lifecycle management policies** in Blob Storage to automatically tier objects from Hot → Cool → Cold → Archive as they age
- Delete unattached managed disks (snapshots from deleted VMs are a common waste source)
- Use the correct redundancy level — LRS is 3× cheaper than GRS; only use GRS where BCDR requirements demand it

### Networking costs
- Keep traffic within the same region to avoid cross-region egress charges
- Use **Private Endpoints** to route traffic within the Azure backbone (cheaper than internet egress)
- Use **Azure CDN or Azure Front Door** to cache content at the edge — reduces origin egress
- Be aware of **NAT Gateway** ($0.045/hour + $0.045/GB processed) — evaluate vs direct outbound

### Delete unused resources
- Unattached public IP addresses, empty App Service plans, idle Application Gateways, and orphaned load balancers all incur charges
- Azure Advisor surfaces orphaned resources
- Use resource tags and Azure Policy to enforce tagging — untagged resources are a sign of unmanaged sprawl

### Serverless and consumption pricing
- Azure Functions Consumption plan: first 1 million executions/month free; only pay when code runs
- Azure Container Apps: scale to zero — no charges when idle
- Cosmos DB serverless: pay per RU consumed, no minimum — ideal for dev/test or low-traffic applications

## Azure Advisor — Cost Recommendations

Azure Advisor is the built-in recommendation engine across five pillars: Cost, Security, Reliability, Operational Excellence, and Performance.

**Cost recommendations include:**

| Recommendation | Description |
|---|---|
| Right-size or shut down VMs | VMs with low CPU/memory utilisation |
| Buy Reserved Instances | Identifies VMs running consistently that would benefit from RIs |
| Buy Savings Plans | Suggests a Savings Plan commitment based on usage |
| Eliminate unprovisioned ExpressRoute circuits | Circuits in Provider Not Provisioned state still incur charges |
| Delete or reconfigure idle load balancers | Load balancers with no backend pool members |
| Reduce costs by deleting unassociated public IPs | Static public IPs not attached to any resource |
| Use Azure Hybrid Benefit | Identifies eligible VMs not yet using AHB |

Each recommendation shows the estimated monthly savings. You can act on recommendations directly from Advisor, snooze them, or dismiss them.

Advisor Score gives an overall percentage (0–100) for each pillar. For the Cost pillar, implementing all active recommendations would bring you to 100%.

## Tagging for Cost Allocation

Tags are key-value pairs applied to Azure resources and resource groups. They are essential for cost allocation:

- `environment: production` / `environment: dev`
- `team: platform` / `team: data`
- `project: checkout-service`
- `cost-centre: CC-1234`

In Cost Analysis, you can group and filter by tag values. For example, view all spending by the `team` tag to produce a per-team cost breakdown. Azure Policy can enforce that all resources have mandatory tags — any resource missing the `cost-centre` tag is flagged as non-compliant.

## Billing Structure — EA, MCA, CSP

### Enterprise Agreement (EA)
For large organisations committing to a minimum annual Azure spend. Features:
- Azure Prepayment (formerly Monetary Commitment): pay upfront for Azure credits at a discounted rate
- Department and account hierarchy for charge-back
- Price sheet negotiated based on commitment level
- Managed through the EA Portal (ea.azure.com)

### Microsoft Customer Agreement (MCA)
The modern billing agreement replacing EA for most customers. Simpler structure:
- Billing Account → Billing Profile → Invoice Section
- Invoice sections map to departments or cost centres
- Managed entirely through the Azure portal
- Supports pay-as-you-go and reservation purchases

### Cloud Solution Provider (CSP)
Purchasing Azure through a Microsoft partner. The partner handles billing, support, and may bundle additional services. Costs may be higher than direct EA pricing but the partner provides managed service value.

### Management Groups and Subscriptions for cost control

```
Root Management Group
├── Platform MG          (shared services — hub VNet, logging, identity)
├── Landing Zones MG
│   ├── Production Sub   (prod workloads)
│   └── Development Sub  (dev/test workloads — Dev/Test pricing eligible)
└── Sandbox MG           (experimentation — budget limit enforced)
```

Separate subscriptions for production and non-production enable:
- Different budget limits
- Dev/Test subscription pricing (requires Visual Studio subscriptions)
- Clear cost isolation between environments
- Different Azure Policy assignments (relaxed in sandbox)

## Code Demo — Cost Management APIs

In [ ]:
# pip install azure-mgmt-costmanagement azure-mgmt-consumption azure-mgmt-advisor
# pip install azure-identity

from azure.identity import DefaultAzureCredential
from azure.mgmt.costmanagement import CostManagementClient
from azure.mgmt.costmanagement.models import (
    QueryDefinition, QueryTimePeriod, QueryDataset,
    QueryAggregation, QueryGrouping, TimeframeType
)
from datetime import datetime, timedelta
import os

credential = DefaultAzureCredential()
subscription_id = os.environ["AZURE_SUBSCRIPTION_ID"]

cost_client = CostManagementClient(credential)

scope = f"/subscriptions/{subscription_id}"

# --- Query cost by service for the last 30 days ---
end_date = datetime.utcnow()
start_date = end_date - timedelta(days=30)

query = QueryDefinition(
    type="ActualCost",
    timeframe=TimeframeType.CUSTOM,
    time_period=QueryTimePeriod(
        from_property=start_date,
        to=end_date
    ),
    dataset=QueryDataset(
        granularity="Monthly",
        aggregation={
            "totalCost": QueryAggregation(name="Cost", function="Sum")
        },
        grouping=[
            QueryGrouping(type="Dimension", name="ServiceName")
        ]
    )
)

result = cost_client.query.usage(scope, query)

# Columns: [Cost, ServiceName, Currency]
col_names = [col.name for col in result.columns]
print(f"Columns: {col_names}")
print("\nTop services by cost (last 30 days):")

rows = sorted(result.rows, key=lambda r: r[0], reverse=True)
for row in rows[:10]:
    cost = row[0]
    service = row[1]
    currency = row[2]
    print(f"  {service:<45} {cost:>10.2f} {currency}")

In [ ]:
# --- Query daily cost by resource group ---
daily_query = QueryDefinition(
    type="ActualCost",
    timeframe=TimeframeType.MONTH_TO_DATE,
    dataset=QueryDataset(
        granularity="Daily",
        aggregation={
            "totalCost": QueryAggregation(name="Cost", function="Sum")
        },
        grouping=[
            QueryGrouping(type="Dimension", name="ResourceGroupName")
        ]
    )
)

daily_result = cost_client.query.usage(scope, daily_query)
col_names = [col.name for col in daily_result.columns]
print(f"Columns: {col_names}")
print("\nDaily cost by resource group (MTD, first 10 rows):")
for row in daily_result.rows[:10]:
    print(f"  {str(row[1]):<12} {row[2]:<30} {row[0]:>8.4f} {row[3]}")

In [ ]:
# --- Create a monthly budget with email alert ---
from azure.mgmt.costmanagement.models import (
    Budget, BudgetTimePeriod, BudgetFilter,
    Notification, BudgetOperatorType, CategoryType
)

budget = Budget(
    category=CategoryType.COST,
    amount=500.0,           # $500/month budget
    time_grain="Monthly",
    time_period=BudgetTimePeriod(
        start_date=datetime(2025, 1, 1),
        end_date=datetime(2026, 12, 31)
    ),
    notifications={
        "Actual_80Percent": Notification(
            enabled=True,
            operator=BudgetOperatorType.GREATER_THAN,
            threshold=80.0,
            contact_emails=["finance-team@example.com", "platform-team@example.com"],
            threshold_type="Actual"
        ),
        "Forecasted_110Percent": Notification(
            enabled=True,
            operator=BudgetOperatorType.GREATER_THAN,
            threshold=110.0,
            contact_emails=["finance-team@example.com"],
            threshold_type="Forecasted"
        )
    }
)

created_budget = cost_client.budgets.create_or_update(
    scope=scope,
    budget_name="monthly-500-budget",
    parameters=budget
)
print(f"Budget created: {created_budget.name}")
print(f"  Amount: ${created_budget.amount:.2f} / {created_budget.time_grain}")
print(f"  Notifications: {list(created_budget.notifications.keys())}")

In [ ]:
# --- Azure Advisor cost recommendations ---
from azure.mgmt.advisor import AdvisorManagementClient
from azure.mgmt.advisor.models import ResourceRecommendationBase

advisor_client = AdvisorManagementClient(credential, subscription_id)

# List all cost recommendations
recommendations = advisor_client.recommendations.list()

cost_recs = [
    r for r in recommendations
    if r.category and r.category.lower() == "cost"
]

print(f"Cost recommendations: {len(cost_recs)}")
for rec in cost_recs[:5]:
    savings = None
    if rec.extended_properties:
        savings = rec.extended_properties.get("savingsAmount")
    resource_name = rec.resource_metadata.resource_id.split("/")[-1] if rec.resource_metadata else "N/A"
    print(f"\n  Resource: {resource_name}")
    print(f"  Impact:   {rec.impact}")
    print(f"  Problem:  {rec.short_description.problem if rec.short_description else 'N/A'}")
    print(f"  Solution: {rec.short_description.solution if rec.short_description else 'N/A'}")
    if savings:
        print(f"  Est. monthly savings: ${float(savings):.2f}")

In [ ]:
# --- List Reserved Instance recommendations ---
from azure.mgmt.consumption import ConsumptionManagementClient

consumption_client = ConsumptionManagementClient(credential, subscription_id)

# Get RI purchase recommendations for VMs (30-day lookback, 1-year term)
ri_recs = consumption_client.reservation_recommendations.list(
    scope=f"/subscriptions/{subscription_id}",
    filter="properties/lookBackPeriod eq 'Last30Days' and properties/scope eq 'Single'"
)

print("Reserved Instance recommendations:")
for rec in ri_recs:
    print(f"\n  SKU: {rec.sku_name}")
    print(f"  Location: {rec.location}")
    print(f"  Recommended quantity: {rec.recommended_quantity}")
    if hasattr(rec, 'net_savings') and rec.net_savings:
        print(f"  Net annual savings: ${rec.net_savings:.2f}")
    if hasattr(rec, 'total_cost_with_reserved_instances') and rec.total_cost_with_reserved_instances:
        print(f"  Cost with RI: ${rec.total_cost_with_reserved_instances:.2f}")

In [ ]:
# --- Tag compliance check: find resources missing a required tag ---
from azure.mgmt.resource import ResourceManagementClient

resource_client = ResourceManagementClient(credential, subscription_id)

REQUIRED_TAGS = ["environment", "team", "cost-centre"]

print("Resources missing required tags:")
missing_count = 0

for resource in resource_client.resources.list():
    tags = resource.tags or {}
    missing = [t for t in REQUIRED_TAGS if t not in tags]
    if missing:
        missing_count += 1
        print(f"  {resource.name:<40} type={resource.type.split('/')[-1]:<20} missing={missing}")
        if missing_count >= 10:
            print("  ... (truncated)")
            break

print(f"\nTotal resources missing required tags (first 10 shown): {missing_count}")

## Summary

| Concept | Key Detail |
|---|---|
| **Pricing model** | Pay-as-you-go by default; egress costs money, ingress is free |
| **Reserved Instances** | 1 or 3-year commitment → up to 72% off PAYG; best for 24/7 steady-state workloads |
| **Savings Plans** | Commit to hourly spend → up to 65% off; flexible across VM sizes and regions |
| **Spot VMs** | Up to 90% off; evictable with 30s notice — for batch/CI/fault-tolerant workloads |
| **Azure Hybrid Benefit** | Use existing Windows/SQL licences on Azure — up to 85% off |
| **Cost Analysis** | View spending by scope, service, resource group, tag |
| **Budgets** | Set thresholds; alert on actual or forecasted overspend; trigger Action Groups |
| **Azure Advisor** | Recommendations for right-sizing, RIs, AHB, orphaned resources |
| **Tagging** | Required for cost allocation; enforce with Azure Policy |
| **Dev/Test pricing** | Separate non-prod subscriptions eligible for discounted Dev/Test rates |
| **Billing accounts** | EA for large enterprises; MCA for modern agreements; CSP through partners |
| **Cost exports** | Schedule CSV exports to Storage Account for Power BI / internal allocation |